# Audio Steganography

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
import IPython.display as ipd

In [2]:
sr, x = wavfile.read("submarine.wav")
ipd.Audio(x, rate=sr)

/tmp/ipykernel_39080/4085373286.py:1: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, x = wavfile.read("submarine.wav")


## LSB Data Hiding

In [ ]:
## Encode a string as bits
string_to_hide = "Héllo world"
hidden_bytes = string_to_hide.encode("utf8")
print(hidden_bytes)
hidden_bits = "".join(f"{b:08b}" for b in hidden_bytes)
print(hidden_bits)

## Hide the bits in the audio LSB and write to disk
# -32768, 32767
x_hide = x - (x % 2)
hidden_bits = np.array([int(b) for b in hidden_bits], dtype=np.int16)
x_hide[:len(hidden_bits)] += hidden_bits
wavfile.write("hidden.wav", sr, x_hide)

## Load audio with hidden information and decode that information
_, y = wavfile.read("hidden.wav")
bits = (y % 2)[0:12*8]
bit_rows = np.reshape(bits, (bits.size//8, 8))
decoded_bytes = bytearray([int("".join([str(c) for c in row]), 2) for row in bit_rows])
decoded_string = decoded_bytes.decode("utf8")
print(decoded_string)

ipd.Audio(x_hide, rate=sr)

In [ ]:
## Encode a file as bits
with open("yellowsubmarine.png", "rb") as fin:
    hidden_bytes = fin.read()
hidden_bits = "".join(f"{b:08b}" for b in hidden_bytes)

## Hide the bits in the audio LSB and write to disk
# -32768, 32767
x_hide = x - (x % 2)
hidden_bits = np.array([int(b) for b in hidden_bits], dtype=np.int16)
x_hide[:len(hidden_bits)] += hidden_bits
wavfile.write("hidden.wav", sr, x_hide)

## Load audio with hidden information and decode that information
_, y = wavfile.read("hidden.wav")
bits = (y % 2)
bit_rows = np.reshape(bits, (bits.size//8, 8))
decoded_bytes = bytearray([int("".join([str(c) for c in row]), 2) for row in bit_rows])
with open("decoded.png", "wb") as fout:
    fout.write(decoded_bytes)

ipd.Audio(x_hide, rate=sr)

## Compression

In [ ]:
!ffmpeg -i hidden.wav hidden.mp3

In [ ]:
!ffmpeg -i hidden.mp3 hidden_mp3.wav

In [ ]:
_, y = wavfile.read("hidden_mp3.wav")
bits = (y % 2)
bit_rows = np.reshape(bits, (bits.size//8, 8))
decoded_bytes = bytearray([int("".join([str(c) for c in row]), 2) for row in bit_rows])
with open("decoded.png", "wb") as fout:
    fout.write(decoded_bytes)

## Echo Hiding

In [ ]:
ipd.Audio(x, rate=sr)

In [3]:
sr, x = wavfile.read("submarine.wav")
x = np.array(x, dtype=float)

string_to_hide = "Héllo world"
hidden_bytes = string_to_hide.encode("utf8")
hidden_bits = "".join(f"{b:08b}" for b in hidden_bytes)
hidden_bits = np.array([int(b) for b in hidden_bits], dtype=np.int16)

/tmp/ipykernel_39080/800492173.py:1: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, x = wavfile.read("submarine.wav")


In [15]:
win = 4096
hop = win // 2
t = np.linspace(0, 2*np.pi, win)
hann = 0.5*(1 - np.cos(t))

lag0 = 75
lag1 = 50
alpha = 0.2
x_echo0 = x.copy()
x_echo0[lag0:] += alpha * x[:-lag0]

x_echo1 = x.copy()
x_echo1[lag1:] += alpha * x[:-lag1]

zero_mix = np.zeros(len(x))
one_mix = np.zeros(len(x))

for i, b in enumerate(hidden_bits):
    if b == 0:
        zero_mix[hop*i:hop*i+win] += hann
    else:
        one_mix[hop*i:hop*i+win] += hann

original_mix = 1 - (one_mix + zero_mix)
x_hidden = x_echo0 * zero_mix + x_echo1 * one_mix + original_mix * x


ipd.Audio(x_hidden, rate=sr)

"Impulse response":
$$
h[n] = 
\left\{
  \begin{array}{ll}
    1 & n=0,\\
    \alpha & n = \text{lag}, \\
    0 & \text{otherwise}
  \end{array}
\right.
$$

\* is convolution

y[n] = x[n] * h[n]

FT[x*h] = FT[x] FT[h]  (Convolution/Multiplication Duality)

log(AB) = log(A) + log(B)

log(FT[x*h]) = log(FT[x]) + log(FT[h])  NOTE: Complex log


log(|FT[x*h]|) ~= log(|FT[x]|) + log(|FT[h]|)

iFT(log(|FT[x*h]|) ~= log(|FT[x]|) + log(|FT[h]|))

In [16]:
wavfile.write("hidden_echo.wav", sr, x_hidden/32768)
!rm hidden_echo.mp3
!ffmpeg -i hidden_echo.wav hidden_echo.mp3
!rm hidden_echo_mp3.wav
!ffmpeg -i hidden_echo.mp3 hidden_echo_mp3.wav
_, x_hidden_mp3 = wavfile.read("hidden_echo_mp3.wav")

ffmpeg version 8.0.1-3ubuntu2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-13ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-pocketsphinx --disable-libcaca --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml

In [17]:
n_win = 1 + (len(x_hidden_mp3)-win)//hop
recovered_bits = np.zeros(n_win, dtype=int)
for i in range(n_win):
    x_hidden_win = x_hidden_mp3[hop*i:hop*i+win] * hann
    cepstrum = np.fft.irfft(np.log(np.abs(np.fft.rfft(x_hidden_win))))
    if cepstrum[lag1] > cepstrum[lag0]:
        recovered_bits[i] = 1

recovered_bits = recovered_bits[:len(hidden_bits)]

n_err = np.sum(np.abs(hidden_bits - recovered_bits))
berr = n_err / len(hidden_bits)
print(n_err, berr)

print(hidden_bits[:20])
print(recovered_bits[:20])

bit_rows = np.reshape(recovered_bits, (recovered_bits.size//8, 8))
decoded_bytes = bytearray([int("".join([str(c) for c in row]), 2) for row in bit_rows])
decoded_string = decoded_bytes.decode("utf8")
print(decoded_string)


11 0.11458333333333333
[0 1 0 0 1 0 0 0 1 1 0 0 0 0 1 1 1 0 1 0]
[0 1 1 1 1 1 0 0 1 1 1 1 0 1 1 1 1 0 1 1]


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf7 in position 1: invalid start byte